# Day 40 — Practice 2 & 3: AdventureWorks → HDFS Parquet + Partisi

**Tujuan:** Extract data dari PostgreSQL AdventureWorks, simpan ke HDFS sebagai Parquet, dan terapkan partisi.

Alur:
```
PostgreSQL (AdventureWorks)  →  Spark  →  HDFS Parquet (Data Lake)
                                                ↓
                                    /datalake/raw/sales_orders/
                                        year=2011/
                                        year=2012/
                                        year=2013/
                                        year=2014/
```

**Konsep yang dipraktikkan:**
- JDBC read dari PostgreSQL
- Tulis ke HDFS sebagai Parquet (columnar format)
- Partisi berdasarkan kolom tertentu
- Manfaat partisi: **partition pruning** saat query

## Setup SparkSession

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/work/jars/postgresql-42.7.3.jar pyspark-shell'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AdventureWorks-to-HDFS') \
    .config('spark.sql.parquet.compression.codec', 'snappy') \
    .getOrCreate()

JDBC_URL = "jdbc:postgresql://adventureworks-postgres:5432/postgres"
JDBC_PROPS = {
    "user": "postgres",
    "password": "My_password1",
    "driver": "org.postgresql.Driver"
}
HDFS_BASE = 'hdfs://namenode:9000/datalake/raw'

def read_pg(schema, table, query=None):
    # Pakai double quotes untuk case-sensitive schema/table
    src = query if query else f'"{schema}"."{table}"'
    return spark.read.jdbc(url=JDBC_URL, table=src, properties=JDBC_PROPS)

print(f'Spark {spark.version} ready')

## 1. Eksplorasi Tabel AdventureWorks

In [ ]:
# Lihat semua tabel yang tersedia
df_tables = read_pg(None, None, query="""
    (SELECT table_schema, table_name, 
            pg_size_pretty(pg_total_relation_size(quote_ident(table_schema)||'.'||quote_ident(table_name))) AS size
     FROM information_schema.tables
     WHERE table_type = 'BASE TABLE'
       AND table_schema NOT IN ('pg_catalog','information_schema')
     ORDER BY table_schema, table_name) t
""")

print(f'Total tabel: {df_tables.count()}')
df_tables.show(30, truncate=False)

## 2. Extract Semua Tabel ke HDFS (Tanpa Partisi)

Untuk tabel referensi / dimensi yang tidak terlalu besar, simpan langsung tanpa partisi.

In [ ]:
# Daftar tabel dimensi / referensi
dim_tables = [
    ('Production', 'Product'),
    ('Production', 'ProductCategory'),
    ('Production', 'ProductSubcategory'),
    ('Person',     'Person'),
    ('HumanResources', 'Department'),
    ('HumanResources', 'Employee'),
    ('Sales',      'Customer'),
    ('Sales',      'SalesTerritory'),
]

results = []
for schema, table in dim_tables:
    df = read_pg(schema, table)
    row_count = df.count()
    hdfs_path = f'{HDFS_BASE}/{schema}/{table}'
    
    df.write \
      .mode('overwrite') \
      .parquet(hdfs_path)
    
    results.append((f'{schema}.{table}', row_count, hdfs_path))
    print(f'✓ {schema}.{table:30s} → {row_count:6,} rows → HDFS')

print('\nSemua tabel dimensi berhasil disimpan ke HDFS!')

## 3. Extract Tabel Fakta dengan Partisi

Tabel fakta (transaksi) biasanya sangat besar. Partisi berdasarkan `year` dan `month`
sangat penting agar query lebih cepat (partition pruning).

### 3.1 Sales Order Header — Partisi by Year

In [ ]:
# Baca sales order header
df_orders = read_pg('Sales', 'SalesOrderHeader')

print('=== Schema SalesOrderHeader ===')
df_orders.printSchema()

print(f'\nTotal rows: {df_orders.count():,}')
print('\nDistribusi per tahun:')
df_orders.withColumn('year', F.year('orderdate')) \
         .groupBy('year').count() \
         .orderBy('year').show()

In [ ]:
# Tambahkan kolom partisi: year dan month
df_orders_partitioned = df_orders \
    .withColumn('order_year',  F.year('orderdate')) \
    .withColumn('order_month', F.month('orderdate'))

# Simpan ke HDFS dengan partisi year, month
orders_path = f'{HDFS_BASE}/Sales/SalesOrderHeader'

df_orders_partitioned.write \
    .mode('overwrite') \
    .partitionBy('order_year', 'order_month') \
    .parquet(orders_path)

print(f'✓ SalesOrderHeader disimpan ke HDFS dengan partisi order_year/order_month')
print(f'  Path: {orders_path}')

In [1]:
# Lihat struktur partisi yang terbentuk di HDFS
import subprocess

print('=== Struktur partisi di HDFS ===')
result = subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -ls -R /datalake/raw/Sales/SalesOrderHeader | head -30',
    shell=True, capture_output=True, text=True
)
print(result.stdout)

print('\n=== Ukuran per partisi ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw/Sales/SalesOrderHeader',
    shell=True
)

=== Struktur partisi di HDFS ===
-rw-r--r--   3 jovyan supergroup          0 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/_SUCCESS
drwxr-xr-x   - jovyan supergroup          0 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/order_year=2011
drwxr-xr-x   - jovyan supergroup          0 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/order_year=2011/order_month=10
-rw-r--r--   3 jovyan supergroup      39110 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/order_year=2011/order_month=10/part-00000-62bd6287-92f8-4cfa-8df3-303cae04d20d.c000.snappy.parquet
drwxr-xr-x   - jovyan supergroup          0 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/order_year=2011/order_month=11
-rw-r--r--   3 jovyan supergroup      27341 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHeader/order_year=2011/order_month=11/part-00000-62bd6287-92f8-4cfa-8df3-303cae04d20d.c000.snappy.parquet
drwxr-xr-x   - jovyan supergroup          0 2026-05-11 02:23 /datalake/raw/Sales/SalesOrderHea

CompletedProcess(args='docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw/Sales/SalesOrderHeader', returncode=0)

### 3.2 Sales Order Detail — Partisi by Year

In [ ]:
# Order detail — join dengan header untuk ambil tanggal
df_detail = read_pg('Sales', 'SalesOrderDetail')

# Join dengan header untuk dapat order_year
df_detail_with_year = df_detail \
    .join(df_orders.select('salesorderid', 'orderdate'), 'salesorderid') \
    .withColumn('order_year', F.year('orderdate')) \
    .drop('orderdate')

print(f'Total order detail rows: {df_detail_with_year.count():,}')

detail_path = f'{HDFS_BASE}/Sales/SalesOrderDetail'
df_detail_with_year.write \
    .mode('overwrite') \
    .partitionBy('order_year') \
    .parquet(detail_path)

print(f'✓ salesorderdetail disimpan dengan partisi order_year')

## 4. Demonstrasi Partition Pruning

Kenapa partisi itu penting? Tanpa partisi → Spark scan **semua file**.
Dengan partisi → Spark hanya scan **folder yang relevan**.

In [ ]:
# Baca tabel yang sudah dipartisi
df_hdfs_orders = spark.read.parquet(orders_path)

print('=== Schema setelah baca dari HDFS ===')
df_hdfs_orders.printSchema()
print(f'Total rows: {df_hdfs_orders.count():,}')

In [ ]:
# Query DENGAN filter partisi (CEPAT — hanya baca folder order_year=2013)
print('=== Query dengan filter partisi (order_year=2013) ===')
df_2013 = df_hdfs_orders.filter(F.col('order_year') == 2013)
print(f'Rows tahun 2013: {df_2013.count():,}')

# Lihat physical plan — cek partition filters
print('\n=== Physical Plan (perhatikan PartitionFilters) ===')
df_2013.explain()

In [ ]:
# Query Q1 2014 saja
df_q1_2014 = df_hdfs_orders.filter(
    (F.col('order_year') == 2014) & (F.col('order_month') <= 3)
)
print(f'Orders Q1 2014: {df_q1_2014.count():,}')

df_q1_2014.select('salesorderid', 'orderdate', 'subtotal', 'order_year', 'order_month').show(5)

## 5. Ringkasan Semua File di HDFS

In [2]:
print('=== Semua file di /datalake/raw ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -h /datalake/raw',
    shell=True
)

print('\n=== Total ukuran datalake ===')
subprocess.run(
    'docker exec hadoop-namenode hdfs dfs -du -s -h /datalake',
    shell=True
)

=== Semua file di /datalake/raw ===
29.2 K  87.6 K   /datalake/raw/HumanResources
2.0 M   6.1 M    /datalake/raw/Person
44.5 K  133.6 K  /datalake/raw/Production
9.3 M   27.9 M   /datalake/raw/Sales
92      92       /datalake/raw/config.json
271     271      /datalake/raw/employees.csv
628     628      /datalake/raw/mapreduce_input
567     567      /datalake/raw/mapreduce_output
232     232      /datalake/raw/sales_summary.csv

=== Total ukuran datalake ===
11.4 M  34.1 M  /datalake


CompletedProcess(args='docker exec hadoop-namenode hdfs dfs -du -s -h /datalake', returncode=0)

## Summary

Yang sudah dipraktikkan:
- ✅ Extract semua tabel dari PostgreSQL AdventureWorks via JDBC
- ✅ Simpan ke HDFS sebagai Parquet (compressed dengan Snappy)
- ✅ Partisi tabel fakta berdasarkan `order_year` dan `order_month`
- ✅ Demonstrasi **partition pruning** — Spark hanya baca partisi yang dibutuhkan

**Data Lake Sekarang:**
```
/datalake/raw/
  production/product/
  production/productcategory/
  person/person/
  humanresources/employee/
  sales/salesorderheader/     ← partisi order_year/order_month
  sales/salesorderdetail/     ← partisi order_year
  ...
```

**Next →** `03_hdfs_to_hive.ipynb` — Buat Hive External Table di atas data HDFS ini.